In [1]:
#import libraries
import numpy as np
from qiskit import QuantumCircuit,QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.circuit import Parameter
from scipy.optimize import minimize
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt 
import pandas as pd
from qiskit_ibm_runtime import QiskitRuntimeService, Session,  Options, Sampler
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager 

In [2]:
#noisy sim (ideal sim only has 29 qbits, but our pb uses 50 qbits)
backend=FakeSherbrooke()


options={'simulator':{'seed_simulator':2000}}
pm = generate_preset_pass_manager(backend=backend,optimization_level=1)
sampler = Sampler(mode=backend,options=options)

In [3]:
# DATA GENERATION STEP ------
np.random.seed(42) 

#Function to generate data
def generate_dataset(name, num_sites):
    site_ids = [f"Site_{i+1}" for i in range(num_sites)]
    install_costs = np.random.randint(15000, 50000, size=num_sites)
    population_coverage = np.random.randint(100, 1500, size=num_sites)
    solar_potential = np.round(np.random.uniform(3.5, 6.5, size=num_sites), 2) #amount of solar energy captured at each site
    energy_capacity = np.round(solar_potential * population_coverage * 0.3, 2) #Estimated energy capacity per site
    coordinates = np.random.uniform(low=0.0, high=1.0, size=(num_sites, 2))
    df = pd.DataFrame({
        "Site_ID": site_ids,
        "Installation_Cost_USD": install_costs,
        "Population_Coverage": population_coverage,
        "Solar_Potential_kWh_m2_day": solar_potential,
        "Energy_Capacity_kWh_day": energy_capacity,
        "X_coord": coordinates[:, 0],
        "Y_coord": coordinates[:, 1]
    })
    return df

df = generate_dataset("Ethiopia_Offgrid_Potential", 50)
df

,Site_ID,Installation_Cost_USD,Population_Coverage,Solar_Potential_kWh_m2_day,Energy_Capacity_kWh_day,X_coord,Y_coord
0,Site_1,30795,261,5.41,423.60,0.703019,0.363630
1,Site_2,15860,301,6.16,556.25,0.971782,0.962447
2,Site_3,26284,1095,4.92,1616.22,0.251782,0.497249
3,Site_4,21265,369,3.86,427.30,0.300878,0.284840
4,Site_5,31850,915,5.64,1548.18,0.036887,0.609564
5,Site_6,36962,1394,5.78,2417.20,0.502679,0.051479
6,Site_7,31023,555,5.18,862.47,0.278646,0.908266
7,Site_8,16685,1375,5.81,2396.62,0.239562,0.144895
8,Site_9,15769,1116,4.98,1667.30,0.489453,0.985650
9,Site_10,17433,395,5.07,600.79,0.242055,0.672136


In [4]:
# PROBLEM PARAMETERS ----
c = df["Installation_Cost_USD"].values #ci
P = df["Population_Coverage"].values#Pi
E = df["Energy_Capacity_kWh_day"].values#Ei
n = len(c) # (num_sites)

# Hyperparameters
alpha = 1e-1
gamma = 1e-1
theta = 1e-6
mu = 2
lambda_ = 1e-2

B = 900000 #(budget)
K = 10 #(max_grids)
M = 15000 #(min_population)

l = 1                      # QAOA layers 

In [5]:
#Linear and Quadratic Terms from Objective and Penalty
linear_terms = {}
quadratic_terms = {}

#objective function
for i in range(n):
    for j in range(n):
        if i == j:
            linear_terms[i] = linear_terms.get(i, 0) + 0.5 * (2*theta*B*c[i] + 2*mu*K + 2*lambda_*M *P[i] - c[i] + alpha*P[i] + gamma*E[i])
        else:
            linear_terms[i] = linear_terms.get(i, 0) - 0.25 * (theta * c[i] * c[j] + mu + lambda_ * P[i] * P[j])
            linear_terms[j] = linear_terms.get(i, 0) - 0.25 * (theta * c[i] * c[j] + mu + lambda_ * P[i] * P[j])
        if i !=j:
            quadratic_terms[(i, j)] = quadratic_terms.get((i, j), 0) + 0.25 * (theta*c[i]*c[j] + mu + lambda_*P[i]*P[j])

# print(linear_terms)
# print()
# print(quadratic_terms )

In [6]:
#Construct QAOA circuit
def create_qaoa_circuit(total_qubits, linear_terms, quadratic_terms, n_layers, betas, gammas):
    qc = QuantumCircuit(total_qubits)
    
    # Initialize superposition
    for q in range(total_qubits):
        qc.h(q)

    # Build QAOA layers
    for layer in range(n_layers):
        # Cost Hamiltonian evolution
        gamma = gammas[layer]
        # Linear terms (Z rotations)
        for q, coeff in linear_terms.items():
            qc.rz(2 * gamma * coeff, q)
        # Quadratic terms (ZZ rotations)
        for (q1, q2), coeff in quadratic_terms.items():
            qc.cx(q1, q2)
            qc.rz(2 * gamma * coeff, q2)
            qc.cx(q1, q2)
        # Mixer Mixer Hamiltonian evolution (X rotations)
        beta = betas[layer]
        for q in range(total_qubits):
            qc.rx(2 * beta, q)
            
    qc.barrier()
    qc.measure_all()
    return qc

qc = create_qaoa_circuit(n, linear_terms, quadratic_terms, l, [1]*l,[1]*l)
# qc.draw('mpl')

In [7]:
#Compute energy expectation
# Function to evaluate the cost function for a given bitstring
def compute_cost(bitstring, linear, quadratic):
    cost = 0#K #0.0
    bits = [int(bit) for bit in bitstring[::-1]]  
    # Linear terms
    for q, coeff in linear.items():
        z = 1 - 2 * bits[q]
        cost += coeff * z
    # Quadratic terms
    for (q1, q2), coeff in quadratic.items():
        z1 = 1 - 2 * bits[q1]
        z2 = 1 - 2 * bits[q2]
        cost += coeff * z1 * z2
    return cost


# Objective function for the optimizer

# Storing the cost values over iterations for plotting
cost_values = []

def objective(params, linear, quadratic):
    beta = params[:l]
    gamma = params[l:]
    qc = create_qaoa_circuit(n, linear_terms, quadratic_terms, l, beta, gamma)
    
    isa_circuit = transpile(qc, backend)
    job=sampler.run([isa_circuit])
    job.job_id()
    result = backend.run(isa_circuit, shots=10000).result()
    counts = result.get_counts()

    # Compute the average cost over all measured bitstrings
    avg_cost = 0
    for bitstring, count in counts.items():
        cost = compute_cost(bitstring, linear, quadratic)
        avg_cost += cost * (count / Shots)  
    cost_values.append(avg_cost)
    return avg_cost


In [8]:
#COBYLA

# initial_params = np.array([pi/4] * 2 * p)  # Initial parameters for beta and gamma
initial_params =np.random.uniform(0, np.pi, 2 * l)
# result = minimize(objective(initial_params,linear_terms, quadratic_terms), initial_params, method='COBYLA',options={'maxiter':300})
result = minimize(objective, initial_params, args=(linear_terms, quadratic_terms), method='COBYLA', options={'maxiter': 300})

print(result)
optimal_params = result.x

print("Optimal parameters (beta, gamma):", optimal_params)


Simulation failed and returned the following error message:
ERROR:  [Experiment 0] Insufficient memory to run circuit circuit-47 using the statevector simulator. Required memory: 17179869184M, max memory: 14789M
Simulation failed and returned the following error message:
ERROR:  [Experiment 0] Insufficient memory to run circuit circuit-47-63 using the statevector simulator. Required memory: 17179869184M, max memory: 14789M


QiskitError: 'ERROR:  [Experiment 0] Insufficient memory to run circuit circuit-47 using the statevector simulator. Required memory: 17179869184M, max memory: 14789M ,  ERROR: Insufficient memory to run circuit circuit-47 using the statevector simulator. Required memory: 17179869184M, max memory: 14789M'